# 02 — Multimodal LLMs with LLaVA

**Goal**: Use LLaVA (local multimodal model) to understand images with text.

Multimodal LLMs can process images, audio, and video — not just text.
LLaVA = Large Language and Vision Assistant.

In [ ]:
# First pull the model if not already pulled
# !ollama pull llava

import requests
import json
from IPython.display import display, Image as IPImage

print("Ready")

## 1. What is a Multimodal Model?

```
Text: "What's in this image?"
         │
         v
    ┌──────────┐    ┌──────────┐
    │  Text    │    │  Vision  │
    │ Encoder  │    │ Encoder  │
    └────┬─────┘    └────┬─────┘
         │               │
         └───────┬───────┘
                 │
          ┌──────┴──────┐
          │  LLM Core   │
          │ (Llama/     │
          │  Mistral)   │
          └──────┬──────┘
                 │
          ┌──────┴──────┐
          │  "A cat     │
          │   sitting   │
          │   on a mat" │
          └─────────────┘
```

## 2. Generate and Save a Test Image

We'll create a simple image using matplotlib to test with.

In [ ]:
# Create a simple test image
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 4))
ax.text(0.5, 0.7, "Hello World", fontsize=24, ha='center', transform=ax.transAxes)
ax.text(0.5, 0.3, "This is a test image", fontsize=14, ha='center', transform=ax.transAxes)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.savefig("test_image.png", dpi=100)
plt.close()
print("Test image saved as 'test_image.png'")

## 3. Query LLaVA (Multimodal)

Ollama supports multimodal models natively.

In [ ]:
def query_multimodal(image_path, prompt):
    """Send an image + text prompt to LLaVA via Ollama."""
    import base64
    
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()
    
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "llava",
        "prompt": prompt,
        "images": [image_b64],
        "stream": False,
        "temperature": 0.2
    })
    return resp.json()["response"]

result = query_multimodal("test_image.png", "Describe what you see in this image")
print(f"LLaVA says:\n{result}")

## 4. Use with LangChain

LangChain supports multimodal models too.

In [ ]:
from langchain_community.llms import Ollama

# LangChain's Ollama wrapper works with multimodal models
multimodal_llm = Ollama(model="llava", temperature=0.2)

# For images, Ollama LangChain wrapper takes base64 in prompts
import base64
with open("test_image.png", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

prompt = f"Describe this image: ![image](data:image/png;base64,{img_b64})"
result = multimodal_llm.invoke(prompt)
print(f"LangChain + LLaVA:\n{result}")

## Applications

1. **Image captioning**: Describe photos automatically
2. **Document understanding**: Extract info from PDFs/forms
3. **Visual QA**: Ask questions about diagrams, charts
4. **Multimodal RAG**: Retrieve images + text together